# 04. Inferential models

Tests H1 to H3 (and secondary outcomes) following Table 4 of the proposal. **Dummy data by default; nothing here is a finding until `DATA_DIR` points at real data.**

| Outcome | Primary model | Basis |
| :-- | :-- | :-- |
| T_find (H1) | linear mixed model on log time, participant random intercept | successful trials |
| E (H2) | Poisson GEE (negative-binomial if overdispersed), clustered on participant | successful trials |
| R (H3) | linear mixed model on log time | successful trials with E >= 1 |
| success | logistic GEE | all trials |
| T_clean, SEQ | linear mixed model | secondary |

All include a block-position term (block position and task pair are the same factor in this design). Omnibus pattern effect: Wald test; pairwise contrasts with Holm correction; reference pattern DH.

**Fallback rule (fixed in advance):** if the primary model is not estimable or did not converge, or its residuals fail Shapiro or Levene at p < .01, the decision uses the Friedman test (Wilcoxon follow-ups) instead. Both are always computed.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA_DIR = ROOT / "data" / "dummy"      # <- point this at data/raw/<your export folder> for real data
OUT_DIR = ROOT / "outputs"

from IPython.display import Image, display
from pipeline.prepare import prepare
from pipeline.models import run_models
from pipeline.plots import make_model_figures

prep = prepare(DATA_DIR)
suite = run_models(prep)
suite.write(OUT_DIR)
paths = make_model_figures(suite, OUT_DIR)

## Decision table
One row per outcome. `decision_source` says which test the decision uses; `flag_reasons` explains any fallback. `omnibus_p` is the primary model; `friedman_p` is the fallback.

In [ ]:
cols = ['outcome','hypothesis','primary_model','n_obs','omnibus_p','friedman_p','flagged_for_fallback','decision_source','decision_p','significant']
suite.summary()[cols].round(4)

In [ ]:
suite.summary()[['outcome','flag_reasons','notes']]

## Pairwise contrasts and effect sizes
`effect` is a ratio of geometric means (times), a rate ratio (E) or an odds ratio (success) for the primary model, and a median paired ratio/difference for the Friedman fallback. Read `MM vs DH` as MM relative to DH. Only report pairwise results when the omnibus test is significant.

In [ ]:
pw = suite.pairwise()
pw[pw.is_decision_source].drop(columns='is_decision_source').round(3)

### All primary and fallback contrasts

In [ ]:
pw.drop(columns='is_decision_source').round(3)

## Sensitivity analyses
Do the conclusions hold when block 1 is dropped, when familiarity and prior use are added, and when failures are treated as censored (Cox model)? `agrees_with_primary` compares significance only. Disagreement is a limitation to report.

In [ ]:
suite.sensitivity[['outcome','analysis','n_obs','omnibus_p','significant','agrees_with_primary','note']].round(4)

## Preference ranking (secondary)

In [ ]:
pf = suite.preference
print(f'Friedman chi2 = {pf.chi2:.2f}, p = {pf.p:.3f}, Kendall W = {pf.kendall_w:.2f}, n = {pf.n_participants}')
pf.pairwise.round(3)

## Figures

### F12: pairwise effect sizes
Filled dot = significant after Holm correction. If an interval crosses 1, that pair is not distinguishable.

In [ ]:
display(Image(filename=str(paths['F12_pairwise_effects'])))

### F11: model diagnostics
Q-Q points should follow the line; residuals should show no funnel or pattern against fitted values. Strong departures trigger the fallback.

In [ ]:
display(Image(filename=str(paths['F11_model_diagnostics'])))

## Reading these results honestly
- With N = 15 (30 trials per pattern) power is limited: "not significant" means *not detected*, not *no difference*.
- GEE effects are population-averaged, not participant-specific (a deviation from Table 4's mixed models).
- Block position and task pair are aliased, so a block effect may be learning or task difficulty.
- Failures are excluded from the time and error models; check the sensitivity table and the success rates.
- R uses only trials with errors, so n is small; check `n_obs` before interpreting it.